# 🌊 ARIMA Water Level Imputation
## Lake Tanganyika Project — Phase 1

This notebook fills **missing monthly water level values** for all rivers  
flowing into Lake Tanganyika using the **ARIMA** model.

---

### What is ARIMA? (simple explanation)

| Letter | Meaning | What it does |
|--------|---------|--------------|
| **AR** | Autoregressive | Uses *past values* of the same river to predict |
| **I**  | Integrated | Removes trends so the data becomes stable |
| **MA** | Moving Average | Smooths out random noise |
| **m=12** | Seasonal period | Tells the model seasons repeat every **12 months** |

---

### Notebook structure

1. **Install & import** libraries  
2. **Load** the master dataset  
3. **Explore** missing values per river  
4. **Run ARIMA** to fill missing months  
5. **Visualise** results  
6. **Evaluate** model accuracy  
7. **Save** the imputed dataset  


## 1. Install & Import Libraries

Run the cell below **once** to install the required packages.  
After installation, restart the kernel and continue.


In [ ]:
# Run this cell once, then restart the kernel
# pmdarima  → automatically selects best ARIMA parameters
# statsmodels → fits the final ARIMA/SARIMAX model
%pip install pmdarima statsmodels --quiet


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import os

from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

print("✓ All libraries imported successfully")


## 2. Settings

Change the paths and options below if needed.


In [ ]:
# ── File paths ──────────────────────────────────────────────────────────────
INPUT_FILE  = "master_dataset_inputed.csv"  # your input CSV
OUTPUT_FILE = "arima_imputed_output.csv"    # filled results will be saved here
PLOTS_DIR   = "arima_plots"                 # folder for charts

# ── Column to fill ───────────────────────────────────────────────────────────
# This is the column that still has NaN values after the previous imputation step
TARGET_COLUMN = "water_level_imputed_v2"

# ── ARIMA settings ───────────────────────────────────────────────────────────
MIN_OBS   = 24   # minimum observed months needed to train ARIMA (at least 2 years)
MAX_P     = 3    # maximum autoregressive terms to try (reduce to 2 for speed)
MAX_Q     = 3    # maximum moving-average terms to try  (reduce to 2 for speed)

os.makedirs(PLOTS_DIR, exist_ok=True)
print(f"✓ Settings ready. Plots will be saved to: {PLOTS_DIR}/")


## 3. Load & Explore the Dataset


In [ ]:
df = pd.read_csv(INPUT_FILE)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["river", "date"]).reset_index(drop=True)

print(f"Dataset shape : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Date range    : {df['date'].min().date()}  →  {df['date'].max().date()}")
print(f"Rivers        : {sorted(df['river'].unique())}")
df.head(5)


In [ ]:
# ── Missing value summary per river ─────────────────────────────────────────
summary = []
for river in sorted(df["river"].unique()):
    sub = df[df["river"] == river]
    total    = len(sub)
    observed = sub[TARGET_COLUMN].notna().sum()
    missing  = sub[TARGET_COLUMN].isna().sum()
    pct      = round(missing / total * 100, 1)
    summary.append({"River": river, "Total months": total,
                    "Observed": observed, "Missing": missing,
                    "Missing %": pct})

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
print(f"\nTotal missing months across all rivers: {summary_df['Missing'].sum()}")


In [ ]:
# ── Bar chart: missing months per river ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

colors = ["#C62828" if m > 50 else "#EF6C00" if m > 10 else "#2E7D32"
          for m in summary_df["Missing"]]

bars = ax.bar(summary_df["River"], summary_df["Missing"], color=colors, width=0.6)

for bar, val in zip(bars, summary_df["Missing"]):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_title("Missing water level months per river (after previous imputation)",
             fontsize=13)
ax.set_ylabel("Number of missing months")
ax.set_xlabel("River")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#C62828", label="> 50 missing (high priority)"),
    Patch(facecolor="#EF6C00", label="11–50 missing"),
    Patch(facecolor="#2E7D32", label="0–10 missing (good)"),
]
ax.legend(handles=legend_elements, fontsize=9, loc="upper right")

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/missing_summary.png", dpi=150)
plt.show()
print("✓ Chart saved")


## 4. ARIMA Imputation

### How the gap-filling works

```
River time series (example):
  Jan  Feb  Mar  Apr  May  Jun  Jul  Aug  Sep  Oct
  0.54 0.49 0.55 NaN  NaN  NaN  0.44 0.41 ...

                        ↑ GAP (3 months)

Step 1: Train ARIMA on all observed values BEFORE the gap
        → model learns the seasonal pattern

Step 2: Forecast forward by 3 steps
        → fills Apr, May, Jun with predictions

Step 3: Move to next gap and repeat
```


In [ ]:
def find_missing_gaps(series: pd.Series):
    """
    Finds all continuous blocks of NaN values in a Series.
    Returns a list of (start_index, end_index) tuples.

    Example:
      series = [0.5, NaN, NaN, NaN, 0.4, NaN, 0.6]
      → gaps at (1, 3) and (5, 5)
    """
    gaps = []
    in_gap = False
    start = None

    for i, val in enumerate(series):
        if pd.isna(val) and not in_gap:
            in_gap = True
            start = i
        elif not pd.isna(val) and in_gap:
            gaps.append((start, i - 1))
            in_gap = False

    if in_gap:                        # gap reaches the end of the series
        gaps.append((start, len(series) - 1))

    return gaps


def fit_arima(observed_values: np.ndarray):
    """
    Automatically finds and fits the best ARIMA model.

    Parameters:
        observed_values : array of known water level measurements

    Returns:
        fitted model, (p,d,q) order, (P,D,Q,m) seasonal order
    """
    model = auto_arima(
        observed_values,
        seasonal    = True,           # allow seasonal component
        m           = 12,             # season length = 12 months
        stepwise    = True,           # faster parameter search
        suppress_warnings = True,
        error_action      = "ignore",
        max_p = MAX_P,
        max_q = MAX_Q,
        max_P = 2,
        max_Q = 2,
        information_criterion = "aic",   # pick model with lowest AIC
    )
    return model, model.order, model.seasonal_order


def predict_gap(series: pd.Series, gap_start: int, gap_end: int,
                order: tuple, seasonal_order: tuple):
    """
    Fills one gap using ARIMA trained on data before the gap.

    Returns:
        predictions (array), method name (string)
    """
    n_steps    = gap_end - gap_start + 1
    train_data = series.iloc[:gap_start].dropna().values

    # Fallback: if not enough history, use the column mean
    if len(train_data) < MIN_OBS:
        fallback = series.dropna().mean()
        return np.full(n_steps, fallback), "mean_fallback"

    try:
        fit = SARIMAX(
            train_data,
            order          = order,
            seasonal_order = seasonal_order,
            enforce_stationarity  = False,
            enforce_invertibility = False,
        ).fit(disp=False)

        preds = fit.forecast(steps=n_steps)
        preds = np.maximum(preds, 0)          # water level cannot be negative
        return preds, "arima"

    except Exception as e:
        fallback = series.dropna().mean()
        print(f"      ⚠ ARIMA fit failed: {e} — using mean fallback")
        return np.full(n_steps, fallback), "mean_fallback"


print("✓ Helper functions defined")


### Run imputation on all rivers

⏳ **Expected time:** 5–15 minutes depending on your computer.  
Each river trains a new ARIMA model automatically.


In [ ]:
# ── Add output columns to the dataframe ─────────────────────────────────────
df["wl_arima_imputed"] = df[TARGET_COLUMN].copy()
df["wl_arima_method"]  = "observed"           # track how each value was filled

imputation_log = []   # collect results for the summary table

# ── Main loop: one river at a time ───────────────────────────────────────────
for river in sorted(df["river"].unique()):

    print(f"\n{'─'*55}")
    print(f"  River: {river}")

    river_mask = df["river"] == river
    series     = df.loc[river_mask, TARGET_COLUMN].copy()
    n_missing  = series.isna().sum()

    print(f"  Observed: {series.notna().sum()}   Missing: {n_missing}")

    # ── Skip if nothing is missing ──
    if n_missing == 0:
        print("  ✓ No missing values — skipping")
        imputation_log.append({
            "River": river, "Missing before": 0,
            "Filled by ARIMA": 0, "Filled by mean": 0,
            "Model": "—", "AIC": "—"
        })
        continue

    # ── Skip if too few observations ──
    observed_vals = series.dropna().values
    if len(observed_vals) < MIN_OBS:
        print(f"  ⚠ Only {len(observed_vals)} observed months — "
              f"need at least {MIN_OBS}. Skipping.")
        imputation_log.append({
            "River": river, "Missing before": n_missing,
            "Filled by ARIMA": 0, "Filled by mean": 0,
            "Model": "insufficient data", "AIC": "—"
        })
        continue

    # ── Train ARIMA ──────────────────────────────────────────────────────────
    print("  Fitting ARIMA model...")
    fitted_model, order, seasonal_order = fit_arima(observed_vals)
    aic = round(fitted_model.aic(), 2)
    print(f"  Best model: ARIMA{order} × Seasonal{seasonal_order}  |  AIC={aic}")

    # ── Find and fill each gap ───────────────────────────────────────────────
    gaps = find_missing_gaps(series.reset_index(drop=True))
    print(f"  Gaps found: {len(gaps)}")

    filled_arima = 0
    filled_mean  = 0

    for gap_start, gap_end in gaps:
        n_gap = gap_end - gap_start + 1

        # Get the actual row indices in df for this river
        river_indices = df[river_mask].index
        gap_df_indices = river_indices[gap_start : gap_end + 1]

        # Get dates for display
        dates = df.loc[gap_df_indices, "date"]
        date_str = (f"{dates.iloc[0].strftime('%Y-%m')} → "
                    f"{dates.iloc[-1].strftime('%Y-%m')}")
        print(f"    Filling gap: {date_str}  ({n_gap} months)")

        predictions, method = predict_gap(
            series.reset_index(drop=True),
            gap_start, gap_end, order, seasonal_order
        )

        df.loc[gap_df_indices, "wl_arima_imputed"] = predictions
        df.loc[gap_df_indices, "wl_arima_method"]  = method

        if method == "arima":
            filled_arima += n_gap
        else:
            filled_mean  += n_gap

    print(f"  ✓ Filled: {filled_arima} by ARIMA,  {filled_mean} by mean fallback")
    imputation_log.append({
        "River": river, "Missing before": n_missing,
        "Filled by ARIMA": filled_arima, "Filled by mean": filled_mean,
        "Model": f"ARIMA{order}×{seasonal_order}", "AIC": aic
    })

print(f"\n{'='*55}")
print("All rivers processed!")


In [ ]:
# ── Imputation summary table ─────────────────────────────────────────────────
log_df = pd.DataFrame(imputation_log)
print("IMPUTATION SUMMARY")
print("=" * 70)
print(log_df.to_string(index=False))
print(f"\nTotal months filled: "
      f"{log_df['Filled by ARIMA'].sum() + log_df['Filled by mean'].sum()}")


## 5. Visualise Results

Each chart shows:
- 🔵 **Blue line** → original observed values  
- 🔴 **Red dots** → months filled by ARIMA  
- 🟠 **Orange triangles** → months filled by mean fallback  


In [ ]:
def plot_river(river_name: str, save: bool = True):
    """Plot observed vs ARIMA-filled values for one river."""

    sub = df[df["river"] == river_name].copy()

    fig, ax = plt.subplots(figsize=(14, 4))

    # Blue line: observed values
    obs = sub[sub["wl_arima_method"] == "observed"]
    ax.plot(obs["date"], obs["wl_arima_imputed"],
            color="#1565C0", linewidth=1.3, label="Observed", zorder=2)

    # Red dots: ARIMA-filled
    arima_pts = sub[sub["wl_arima_method"] == "arima"]
    if len(arima_pts) > 0:
        ax.scatter(arima_pts["date"], arima_pts["wl_arima_imputed"],
                   color="#C62828", s=22, zorder=3, label=f"ARIMA filled ({len(arima_pts)})")

    # Orange triangles: mean fallback
    mean_pts = sub[sub["wl_arima_method"] == "mean_fallback"]
    if len(mean_pts) > 0:
        ax.scatter(mean_pts["date"], mean_pts["wl_arima_imputed"],
                   color="#E65100", s=22, marker="^", zorder=3,
                   label=f"Mean fallback ({len(mean_pts)})")

    # Get model info from log
    log_row = log_df[log_df["River"] == river_name]
    model_label = log_row["Model"].values[0] if len(log_row) > 0 else ""

    ax.set_title(f"{river_name} — water level  |  {model_label}", fontsize=12)
    ax.set_ylabel("Water level (m)")
    ax.set_xlabel("Date")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    plt.xticks(rotation=45)
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(True, alpha=0.3, linewidth=0.5)
    fig.tight_layout()

    if save:
        path = f"{PLOTS_DIR}/{river_name}_arima.png"
        plt.savefig(path, dpi=150)
        print(f"  Saved → {path}")

    plt.show()


# Plot all rivers
for river in sorted(df["river"].unique()):
    plot_river(river)


## 6. Evaluate Model Accuracy

To check how good the predictions are, we use a **hold-out test**:

```
All observed months:  ████████████████████████████████░░░░░  
                      ← 85% training data →   ← 15% test →

1. Train ARIMA on the first 85% of observed values
2. Predict the last 15%
3. Compare predictions to the real values
4. Calculate RMSE and MAE
```

**Metrics explained:**
- **RMSE** (Root Mean Square Error) — average prediction error in metres. Lower = better.
- **MAE**  (Mean Absolute Error)    — average absolute error in metres. Lower = better.
- **Reference:** Agaj et al. (2024) reported RMSE = 7.5 cm on a similar river study.


In [ ]:
TEST_FRACTION = 0.15   # hold out last 15% of observed data for testing

eval_results = []

print(f"{'River':<15} {'N test':>7} {'RMSE (m)':>10} {'MAE (m)':>10}  Model")
print("─" * 65)

for river in sorted(df["river"].unique()):
    sub      = df[df["river"] == river]
    observed = sub[sub[TARGET_COLUMN].notna()][TARGET_COLUMN].values

    if len(observed) < MIN_OBS * 2:
        print(f"{river:<15} {'—':>7} {'—':>10} {'—':>10}  not enough data")
        continue

    # Split into train / test
    split      = int(len(observed) * (1 - TEST_FRACTION))
    train_vals = observed[:split]
    test_vals  = observed[split:]

    try:
        m = auto_arima(train_vals, seasonal=True, m=12, stepwise=True,
                       suppress_warnings=True, error_action="ignore",
                       max_p=MAX_P, max_q=MAX_Q, max_P=2, max_Q=2)

        fit = SARIMAX(train_vals,
                      order=m.order, seasonal_order=m.seasonal_order,
                      enforce_stationarity=False,
                      enforce_invertibility=False).fit(disp=False)

        preds = np.maximum(fit.forecast(steps=len(test_vals)), 0)

        rmse = np.sqrt(np.mean((test_vals - preds) ** 2))
        mae  = np.mean(np.abs(test_vals - preds))

        print(f"{river:<15} {len(test_vals):>7} {rmse:>10.4f} {mae:>10.4f}"
              f"  ARIMA{m.order}×{m.seasonal_order}")

        eval_results.append({
            "River": river, "N test": len(test_vals),
            "RMSE (m)": round(rmse, 4), "MAE (m)": round(mae, 4),
            "Model": f"ARIMA{m.order}×{m.seasonal_order}"
        })

    except Exception as e:
        print(f"{river:<15} {'ERROR':>7}  {e}")

if eval_results:
    eval_df = pd.DataFrame(eval_results)
    print("─" * 65)
    print(f"{'Average':<15} {'' :>7} {eval_df['RMSE (m)'].mean():>10.4f}"
          f" {eval_df['MAE (m)'].mean():>10.4f}")
    print("\n(Reference: Agaj et al. 2024 paper → RMSE = 0.075 m on a similar river)")


In [ ]:
# ── Bar chart: RMSE per river ────────────────────────────────────────────────
if eval_results:
    fig, ax = plt.subplots(figsize=(10, 4))

    rivers_eval = [r["River"] for r in eval_results]
    rmse_vals   = [r["RMSE (m)"] for r in eval_results]

    colors = ["#C62828" if v > 0.1 else "#1565C0" for v in rmse_vals]
    bars   = ax.bar(rivers_eval, rmse_vals, color=colors, width=0.6)

    for bar, val in zip(bars, rmse_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)

    # Reference line from the paper
    ax.axhline(y=0.075, color="#2E7D32", linestyle="--", linewidth=1.5,
               label="Reference: Agaj et al. 2024 (RMSE=0.075 m)")

    ax.set_title("ARIMA model accuracy — RMSE per river", fontsize=13)
    ax.set_ylabel("RMSE (metres)")
    ax.set_xlabel("River")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/rmse_comparison.png", dpi=150)
    plt.show()
    print("✓ RMSE chart saved")


## 7. Save the Imputed Dataset

The new CSV contains **two extra columns** compared to the original:

| Column | Meaning |
|--------|---------|
| `wl_arima_imputed` | Water level — observed values kept, missing filled by ARIMA |
| `wl_arima_method`  | How was this month filled? `observed` / `arima` / `mean_fallback` |


In [ ]:
df.to_csv(OUTPUT_FILE, index=False)
print(f"✓ Saved imputed dataset → {OUTPUT_FILE}")
print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nNew columns added:")
print(f"  wl_arima_imputed  — {df['wl_arima_imputed'].notna().sum()} non-null values")
print(f"  wl_arima_method   — value counts:")
print(df["wl_arima_method"].value_counts().to_string())

# Quick check: any remaining NaN in output column?
remaining_nan = df["wl_arima_imputed"].isna().sum()
if remaining_nan == 0:
    print("\n✓ All missing values have been filled!")
else:
    print(f"\n⚠ {remaining_nan} months still missing (check rivers with insufficient data)")


In [ ]:
# Preview a few ARIMA-filled rows
arima_sample = df[df["wl_arima_method"] == "arima"][
    ["river", "date", TARGET_COLUMN, "wl_arima_imputed", "wl_arima_method"]
].head(10)

print("Sample of ARIMA-filled rows:")
print(arima_sample.to_string(index=False))


---

## ✅ Done!

**Output files:**
- 📄 `arima_imputed_output.csv` — full dataset with filled water levels  
- 📊 `arima_plots/missing_summary.png` — bar chart of missing months  
- 📊 `arima_plots/<river>_arima.png` — one chart per river  
- 📊 `arima_plots/rmse_comparison.png` — model accuracy comparison  

**Next steps for Phase 2:**
1. Use `wl_arima_imputed` as the water level column for trend analysis  
2. Build prediction models (LSTM, Random Forest) on the clean dataset  
3. Integrate results into the website  
